# Unsupervised BNN Mixture Model for Multiclass Classification
This notebook demonstrates how to build an unsupervised Bayesian Neural Network (BNN) mixture model for multiclass classification using `BayesInference`. The logic combines the Bayesian Neural Network (BNN) with an unsupervised mixture objective, treating the cluster assignments as latent variables.

We compare the BNN Mixture Model with a standard Dirichlet Process Mixture Model (DPMM) baseline on a synthetic blobs dataset.

## 1. Setup and Data Generation
First, we initialize the model platform and generate the synthetic dataset.

In [ ]:
import numpy as np
from BI import bi, jnp
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

m = bi(platform='cpu')

# Generate Synthetic Data (4 clusters, same as DPMM baseline)
data, true_labels = make_blobs(
    n_samples=500, centers=4, cluster_std=0.8,
    center_box=(-10,10), random_state=101
)
N, D = data.shape

print(f"Data shape: {data.shape}")
print(f"True labels shape: {true_labels.shape}")

plt.scatter(data[:, 0], data[:, 1], c=true_labels, cmap='viridis')
plt.title('Synthetic Data (True Labels)')
plt.show()

## 2. DPMM Baseline
We fit the standard Dirichlet Process Mixture Model (DPMM) to the data as a baseline.

In [ ]:
m.data_on_model = dict(data=data, T=11)
m.fit(m.models.dpmm, num_chains=1)

dpmm_clusters = m.models.dpmm.proportion_of_data_assigned_to_cluster()
print(f"DPMM identified clusters:\n{dpmm_clusters}")

## 3. Unsupervised BNN Mixture Model
Here we define the unsupervised BNN mixture model. The BNN maps the features $X$ to latent assignment probabilities $\theta_i$ for $K$ classes via a softmax layer. These probabilities then mix $K$ multivariate normal components which directly model the features $X$.

In [ ]:
m.data_on_model = dict(data=data)

def bnn_mixture_model(data, K=11, D_H1=10):
    # data shape: (N, D)
    N, D_X = data.shape
    
    # --- BNN Gating Network ---
    # First hidden layer
    w1 = m.bnn.layer_linear(
        data, 
        dist=m.dist.normal(0, 1, name='w1_weight', shape=(D_X, D_H1)),
        activation='tanh'
    )
    
    # Output layer -> Logits for K classes
    w2 = m.bnn.layer_linear(
        w1,
        dist=m.dist.normal(0, 1, name='w2_weight', shape=(D_H1, K))
    )
    
    # Softmax to get mixing probabilities (theta)
    theta = jnp.exp(jax.nn.log_softmax(w2, axis=-1))
    
    # --- Mixture Components ---
    # Component means (K, D)
    mu = m.dist.normal(0, 5, name='mu', shape=(K, D_X))
    
    # Component scales (K, D)
    sigma = m.dist.half_normal(1, name='sigma', shape=(K, D_X))
    
    # Likelihood: The data is a mixture of K multivariate normals,
    # mixed according to the BNN outputs `theta`.
    m.dist.mixture_same_family(
        mixing_distribution=m.distr.Categorical(probs=theta),
        component_distribution=m.distr.Independent(
            m.distr.Normal(mu, sigma), reinterpreted_batch_ndims=1
        ),
        obs=data
    )

# Note: The `jax` numpy import was used above, make sure `jax` is imported.
import jax


In [ ]:
m.fit(bnn_mixture_model, num_chains=1)